# Linear Regression — Predicting Fuel EfficiencyThis notebook applies linear regression methods to the **UCI Auto MPG** dataset, predicting miles-per-gallon from car attributes (cylinders, displacement, horsepower, weight, etc.). It is a classic regression dataset from the StatLib library at Carnegie Mellon (1983).**Models compared:**- **Ordinary Least Squares (OLS)** — minimizes sum of squared residuals- **Ridge regression** — OLS + L2 penalty (shrinks all coefficients)- **Lasso regression** — OLS + L1 penalty (zeros out unimportant features)**Dataset:** 398 rows, 7 numerical features after cleaning. Source: [UCI](https://archive.ics.uci.edu/dataset/9/auto+mpg) (downloaded automatically below).

In [ ]:
import osimport urllib.requestDATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data") if os.path.basename(os.getcwd()) == "notebooks" else "data"os.makedirs(DATA_DIR, exist_ok=True)def download_if_needed(url, filename):    """Download a CSV if it doesn't already exist locally."""    path = os.path.join(DATA_DIR, filename)    if not os.path.exists(path):        print(f"Downloading {filename} from {url}")        urllib.request.urlretrieve(url, path)    return pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LinearRegression, Ridge, Lassofrom sklearn.metrics import mean_squared_error, r2_scoresns.set_style("whitegrid")np.random.seed(42)

## 1. Load the data

In [ ]:
path = download_if_needed(    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/auto-mpg.csv",    "auto-mpg.csv",)cols = ["mpg", "cylinders", "displacement", "horsepower",        "weight", "acceleration", "model_year", "origin", "name"]df = pd.read_csv(path, header=None, names=cols, na_values="?")df = df.drop(columns="name").dropna()print("Shape:", df.shape)df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))df["mpg"].hist(bins=30, ax=ax[0])ax[0].set_title("Target distribution: mpg")ax[0].set_xlabel("Miles per gallon")corr = df.corr(numeric_only=True)["mpg"].drop("mpg").sort_values()corr.plot(kind="barh", ax=ax[1])ax[1].set_title("Correlation with mpg")plt.tight_layout(); plt.show()

As expected, **weight**, **displacement**, and **cylinders** correlate strongly *negatively* with mpg, while **model_year** correlates positively (newer cars are more efficient).

## 3. Train/test split + feature scaling

In [ ]:
X = df.drop(columns="mpg").valuesy = df["mpg"].valuesX_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42)scaler = StandardScaler()X_train_s = scaler.fit_transform(X_train)X_test_s = scaler.transform(X_test)print(f"Train: {X_train_s.shape}, Test: {X_test_s.shape}")

## 4. Compare three regression models

In [ ]:
models = {    "OLS":   LinearRegression(),    "Ridge": Ridge(alpha=1.0),    "Lasso": Lasso(alpha=0.1),}results = []for name, m in models.items():    m.fit(X_train_s, y_train)    preds = m.predict(X_test_s)    results.append({        "Model": name,        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),        "R^2":  r2_score(y_test, preds),    })pd.DataFrame(results).round(4)

## 5. Coefficient inspection

In [ ]:
coefs = pd.DataFrame(    {name: m.coef_ for name, m in models.items()},    index=df.drop(columns="mpg").columns,)coefs.plot(kind="bar", figsize=(10, 4))plt.title("Coefficients across regularization schemes")plt.axhline(0, color="k", lw=0.5)plt.xticks(rotation=45)plt.tight_layout(); plt.show()coefs.round(3)

Note how **Lasso** drives some coefficients toward zero — that's the L1 penalty's feature-selection behavior.

## 6. Predicted vs Actual

In [ ]:
best = LinearRegression().fit(X_train_s, y_train)preds = best.predict(X_test_s)plt.figure(figsize=(6, 6))plt.scatter(y_test, preds, alpha=0.6)lo, hi = y_test.min(), y_test.max()plt.plot([lo, hi], [lo, hi], "r--", label="y = x")plt.xlabel("Actual mpg"); plt.ylabel("Predicted mpg")plt.title("OLS — Predicted vs Actual")plt.legend(); plt.tight_layout(); plt.show()

## Takeaways- All three regularization schemes yield similar performance (R² ≈ 0.83). Auto MPG has a relatively clean linear signal.- **Weight** and **model_year** dominate. Heavier cars use more fuel; newer cars use less.- **Lasso** zeros out one or two features — useful when you want a parsimonious model.- Linear regression is a strong baseline. Beating it would require a non-linear model like a random forest (see notebook 05).